## Predicting Acute Food Insecurity Risk in Kenyan Counties using Climate and Market Signals

**Author:** Data Alchemists

**Program:** Moringa School - Data Science Capstone Project

**Date:** June-July 2026


---

### Project Overview

Kenya's smallholder farmers and pastoralists face persistent food insecurity driven by rainfall failure and soaring staple food prices. This notebook builds a **binary classification model** that predicts whether a geographic zone in Kenya will reach **IPC Phase 3 (Crisis) or worse** in a given month using satellite-derived rainfall anomalies (CHIRPS) and WFP food price records as leading indicators.

The model is designed to support Kenya's **National Drought Management Authority (NDMA)** in issuing early warnings weeks before official IPC classifications are published.

---


### CRISP-DM Structure

This notebook follows the **CRISP-DM (Cross-Industry Standard Process for Data Mining)** framework:

| Phase | Section |
|---|---|
| Business Understanding | Phase 1 |
| Data Understanding | Phase 2 |
| Data Cleaning | Phase 3 |
| Data Preparation | Phase 4 |
| Modeling | Phase 5 |
| Evaluation | Phase 6 |

Each phase opens with a markdown cell explaining **WHY** the phase is necessary not just what is done.

---

### Datasets

| # | Dataset | Source | Rows | Role |
|---|---|---|---|---|
| 1 | FEWS NET IPC Classifications | HDX / FEWS NET | 27,694 | Target variable |
| 2 | CHIRPS Rainfall Indicators | HDX / WFP | 132,678 | Climate features |
| 3 | WFP Food Prices Kenya | HDX / WFP | 26,745 | Market features |

---
### Domain Citations

1. Funk, C. et al. (2015). *The climate hazards infrared precipitation with stations.* Scientific Data, 2, 150066. Justifies CHIRPS choice and rfq interpretation  
2. FAO (2021). *IPC Technical Manual Version 3.1.* Justifies IPC Phase 3 as binary target threshold  
3. FEWS NET (2023). *Kenya Food Security Outlook, Oct 2022–Mar 2023.* Contextualises 2020–2023 crisis spikes



## DATA UNDERSTANDING

In [80]:
#Importing Libraries
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns


In [81]:
#Loading the datasets
ipc_df = pd.read_excel('data/ipcphase.xlsx')
rainfall_df = pd.read_csv('data/ken-rainfall-subnat-full.csv')
food_df = pd.read_csv('data/wfp_food_prices_ken.csv')


In [82]:
# Checking the first 5 rows on ipc dataframe
ipc_df.head()

,row,source_organization,source_document,country,country_code,geographic_group,fewsnet_region,geographic_unit_full_name,geographic_unit_name,unit_type,...,specialization_type,dataseries_specialization_type,data_usage_policy,created,modified,status_changed,collection_status,collection_status_changed,collection_schedule,reporting_date
0,0,FEWS NET,"Food Security Outlook, Kenya",Kenya,KE,Eastern Africa,East Africa,"Aberdare Forest1, Murang'a, Central, Kenya",Aberdare Forest1,fsc_admin,...,/IPCPhase/,/IPCClassification/,Public,2020-11-25T18:47:21,2020-11-25T18:47:21,2020-11-25T18:53:10,Published,2021-01-05T21:16:32,Ad Hoc,2011-07-01
1,1,FEWS NET,"Food Security Outlook, Kenya",Kenya,KE,Eastern Africa,East Africa,"Aberdare Forest1, Murang'a, Central, Kenya",Aberdare Forest1,fsc_admin,...,/IPCPhase/,/IPCClassification/,Public,2020-11-25T18:47:21,2020-11-25T18:47:21,2020-11-25T18:53:10,Published,2021-01-05T21:16:32,Ad Hoc,2011-10-01
2,2,FEWS NET,"Food Security Outlook, Kenya",Kenya,KE,Eastern Africa,East Africa,"Aberdare Forest1, Murang'a, Central, Kenya",Aberdare Forest1,fsc_admin,...,/IPCPhase/,/IPCClassification/,Public,2020-11-25T18:47:21,2020-11-25T18:47:21,2020-11-25T18:53:10,Published,2021-01-05T21:16:32,Ad Hoc,2012-01-01
3,3,FEWS NET,"Food Security Outlook, Kenya",Kenya,KE,Eastern Africa,East Africa,"Aberdare Forest1, Murang'a, Central, Kenya",Aberdare Forest1,fsc_admin,...,/IPCPhase/,/IPCClassification/,Public,2020-11-25T18:47:21,2020-11-25T18:47:21,2020-11-25T18:53:10,Published,2021-01-05T21:16:32,Ad Hoc,2012-04-01
4,4,FEWS NET,"Food Security Outlook, Kenya",Kenya,KE,Eastern Africa,East Africa,"Aberdare Forest1, Murang'a, Central, Kenya",Aberdare Forest1,fsc_admin,...,/IPCPhase/,/IPCClassification/,Public,2020-11-25T18:47:21,2020-11-25T18:47:21,2020-11-25T18:53:10,Published,2021-01-05T21:16:31,Ad Hoc,2012-07-01


In [83]:
#Checking the first 5 rows on rainfall dataframe
rainfall_df.head()

,date,adm_level,adm_id,PCODE,n_pixels,rfh,rfh_avg,r1h,r1h_avg,r3h,r3h_avg,rfq,r1q,r3q,version
0,1981-01-01,1,51325,KE019,427.0,7.372365,15.759407,NaN,NaN,NaN,NaN,59.598840,NaN,NaN,final
1,1981-01-11,1,51325,KE019,427.0,4.325527,19.294770,NaN,NaN,NaN,NaN,38.384920,NaN,NaN,final
2,1981-01-21,1,51325,KE019,427.0,5.569087,16.265417,17.266980,51.319595,NaN,NaN,49.700817,39.536823,NaN,final
3,1981-02-01,1,51325,KE019,427.0,5.882904,12.719282,15.777517,48.279470,NaN,NaN,61.418427,38.997230,NaN,final
4,1981-02-11,1,51325,KE019,427.0,17.180328,18.768618,28.632318,47.753320,NaN,NaN,93.317700,63.753933,NaN,final


In [84]:
#Checking the first 5 rows on food dataframe
food_df.head()

,date,admin1,admin2,market,market_id,latitude,longitude,category,commodity,commodity_id,unit,priceflag,pricetype,currency,price,usdprice
0,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize,51,KG,actual,Wholesale,KES,16.13,0.22
1,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,cereals and tubers,Maize (white),67,90 KG,actual,Wholesale,KES,1480.00,20.58
2,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans,50,KG,actual,Wholesale,KES,33.63,0.47
3,2006-01-15,Coast,Mombasa,Mombasa,191,-4.05,39.67,pulses and nuts,Beans (dry),262,90 KG,actual,Wholesale,KES,3246.00,45.15
4,2006-01-15,Eastern,Kitui,Kitui,187,-1.37,38.02,cereals and tubers,Maize (white),67,KG,actual,Retail,KES,17.00,0.24


## Checking the df info

In [85]:
ipc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27694 entries, 0 to 27693
Data columns (total 42 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   row                             27694 non-null  int64  
 1   source_organization             27694 non-null  object 
 2   source_document                 27694 non-null  object 
 3   country                         27694 non-null  object 
 4   country_code                    27694 non-null  object 
 5   geographic_group                27694 non-null  object 
 6   fewsnet_region                  27694 non-null  object 
 7   geographic_unit_full_name       27694 non-null  object 
 8   geographic_unit_name            27694 non-null  object 
 9   unit_type                       27694 non-null  object 
 10  fnid                            27694 non-null  object 
 11  classification_scale            27694 non-null  object 
 12  scenario_name                   

In [86]:
rainfall_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132678 entries, 0 to 132677
Data columns (total 15 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   date       132678 non-null  object 
 1   adm_level  132678 non-null  int64  
 2   adm_id     132678 non-null  int64  
 3   PCODE      132678 non-null  object 
 4   n_pixels   132678 non-null  float64
 5   rfh        132678 non-null  float64
 6   rfh_avg    132678 non-null  float64
 7   r1h        132516 non-null  float64
 8   r1h_avg    132516 non-null  float64
 9   r3h        132030 non-null  float64
 10  r3h_avg    132030 non-null  float64
 11  rfq        132678 non-null  float64
 12  r1q        132516 non-null  float64
 13  r3q        132030 non-null  float64
 14  version    132678 non-null  object 
dtypes: float64(10), int64(2), object(3)
memory usage: 15.2+ MB


In [87]:
food_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26745 entries, 0 to 26744
Data columns (total 16 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          26745 non-null  object 
 1   admin1        26677 non-null  object 
 2   admin2        26677 non-null  object 
 3   market        26745 non-null  object 
 4   market_id     26745 non-null  int64  
 5   latitude      26677 non-null  float64
 6   longitude     26677 non-null  float64
 7   category      26745 non-null  object 
 8   commodity     26745 non-null  object 
 9   commodity_id  26745 non-null  int64  
 10  unit          26745 non-null  object 
 11  priceflag     26745 non-null  object 
 12  pricetype     26745 non-null  object 
 13  currency      26745 non-null  object 
 14  price         26745 non-null  float64
 15  usdprice      26745 non-null  float64
dtypes: float64(4), int64(2), object(10)
memory usage: 3.3+ MB


## Checking for missing values in the df

In [88]:
ipc_df.isnull().sum()

row                                   0
source_organization                   0
source_document                       0
country                               0
country_code                          0
geographic_group                      0
fewsnet_region                        0
geographic_unit_full_name             0
geographic_unit_name                  0
unit_type                             0
fnid                                  0
classification_scale                  0
scenario_name                         0
preference_rating                     0
is_allowing_for_assistance            0
projection_start                      0
projection_end                        0
status                                0
value                                 0
pct_phase3                        27694
pct_phase4                        27694
pct_phase5                        27694
description                           0
id                                    0
datacollectionperiod                  0


In [89]:
rainfall_df.isnull().sum()

date           0
adm_level      0
adm_id         0
PCODE          0
n_pixels       0
rfh            0
rfh_avg        0
r1h          162
r1h_avg      162
r3h          648
r3h_avg      648
rfq            0
r1q          162
r3q          648
version        0
dtype: int64

In [90]:
food_df.isnull().sum()

date             0
admin1          68
admin2          68
market           0
market_id        0
latitude        68
longitude       68
category         0
commodity        0
commodity_id     0
unit             0
priceflag        0
pricetype        0
currency         0
price            0
usdprice         0
dtype: int64

## Checking for duplicates

In [91]:
ipc_df.duplicated().sum()

0

In [92]:
rainfall_df.duplicated().sum()

0

In [93]:
food_df.duplicated().sum()

0

## Finding the number of rows and columns in each dataset


In [94]:
ipc_df.shape

(27694, 42)

In [95]:
rainfall_df.shape

(132678, 15)

In [96]:
food_df.shape

(26745, 16)

## Checking the column names for each dataset

In [97]:
ipc_df.columns

Index(['row', 'source_organization', 'source_document', 'country',
       'country_code', 'geographic_group', 'fewsnet_region',
       'geographic_unit_full_name', 'geographic_unit_name', 'unit_type',
       'fnid', 'classification_scale', 'scenario_name', 'preference_rating',
       'is_allowing_for_assistance', 'projection_start', 'projection_end',
       'status', 'value', 'pct_phase3', 'pct_phase4', 'pct_phase5',
       'description', 'id', 'datacollectionperiod', 'datacollection',
       'scenario', 'geographic_unit', 'datasourceorganization',
       'datasourcedocument', 'dataseries', 'dataseries_name',
       'specialization_type', 'dataseries_specialization_type',
       'data_usage_policy', 'created', 'modified', 'status_changed',
       'collection_status', 'collection_status_changed', 'collection_schedule',
       'reporting_date'],
      dtype='object')

In [98]:
rainfall_df.columns

Index(['date', 'adm_level', 'adm_id', 'PCODE', 'n_pixels', 'rfh', 'rfh_avg',
       'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version'],
      dtype='object')

In [99]:
food_df.columns

Index(['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude',
       'longitude', 'category', 'commodity', 'commodity_id', 'unit',
       'priceflag', 'pricetype', 'currency', 'price', 'usdprice'],
      dtype='object')

## DATA CLEANING

In [100]:
# Date handling
from datetime import datetime

# Machine Learning
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer


In [101]:
# Standardizing the column names
# Ensuring that all columns are in lowercase
ipc_df.columns = ipc_df.columns.str.lower().str.strip().str.replace(" ", "_")
rainfall_df.columns = rainfall_df.columns.str.lower().str.strip().str.replace(" ", "_")
food_df.columns = food_df.columns.str.lower().str.strip().str.replace(" ", "_")

In [102]:
# Checking our standardized columns
ipc_df.columns

Index(['row', 'source_organization', 'source_document', 'country',
       'country_code', 'geographic_group', 'fewsnet_region',
       'geographic_unit_full_name', 'geographic_unit_name', 'unit_type',
       'fnid', 'classification_scale', 'scenario_name', 'preference_rating',
       'is_allowing_for_assistance', 'projection_start', 'projection_end',
       'status', 'value', 'pct_phase3', 'pct_phase4', 'pct_phase5',
       'description', 'id', 'datacollectionperiod', 'datacollection',
       'scenario', 'geographic_unit', 'datasourceorganization',
       'datasourcedocument', 'dataseries', 'dataseries_name',
       'specialization_type', 'dataseries_specialization_type',
       'data_usage_policy', 'created', 'modified', 'status_changed',
       'collection_status', 'collection_status_changed', 'collection_schedule',
       'reporting_date'],
      dtype='object')

In [103]:
rainfall_df.columns

Index(['date', 'adm_level', 'adm_id', 'pcode', 'n_pixels', 'rfh', 'rfh_avg',
       'r1h', 'r1h_avg', 'r3h', 'r3h_avg', 'rfq', 'r1q', 'r3q', 'version'],
      dtype='object')

In [104]:
food_df.columns

Index(['date', 'admin1', 'admin2', 'market', 'market_id', 'latitude',
       'longitude', 'category', 'commodity', 'commodity_id', 'unit',
       'priceflag', 'pricetype', 'currency', 'price', 'usdprice'],
      dtype='object')

In [105]:
# Changing the date columns to datetime
ipc_df['reporting_date'] = pd.to_datetime(ipc_df['reporting_date'])
rainfall_df['date'] = pd.to_datetime(rainfall_df['date'])
food_df['date'] = pd.to_datetime(food_df['date'])

print(ipc_df['reporting_date'].head())
print(rainfall_df['date'].head())
print(food_df['date'].head())

0   2011-07-01
1   2011-10-01
2   2012-01-01
3   2012-04-01
4   2012-07-01
Name: reporting_date, dtype: datetime64[ns]
0   1981-01-01
1   1981-01-11
2   1981-01-21
3   1981-02-01
4   1981-02-11
Name: date, dtype: datetime64[ns]
0   2006-01-15
1   2006-01-15
2   2006-01-15
3   2006-01-15
4   2006-01-15
Name: date, dtype: datetime64[ns]


In [107]:
print("IPC:")
print(ipc_df['reporting_date'].head())

print("\nRAINFALL:")
print(rainfall_df['date'].head())

print("\nFOOD:")
print(food_df['date'].head())

IPC:
0   2011-07-01
1   2011-10-01
2   2012-01-01
3   2012-04-01
4   2012-07-01
Name: reporting_date, dtype: datetime64[ns]

RAINFALL:
0   1981-01-01
1   1981-01-11
2   1981-01-21
3   1981-02-01
4   1981-02-11
Name: date, dtype: datetime64[ns]

FOOD:
0   2006-01-15
1   2006-01-15
2   2006-01-15
3   2006-01-15
4   2006-01-15
Name: date, dtype: datetime64[ns]
